### Importing Libraries and Initializing Web Driver

In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time
import re # Still needed for other potential parsing, but not for Shorts check

# For automatically managing ChromeDriver
from webdriver_manager.chrome import ChromeDriverManager

# Configure Chrome options
options = Options()
# options.add_argument("--headless=new") # Keep headless for production, remove for debugging visual
options.add_argument("--disable-extensions")
options.add_argument("--disable-notifications")
options.add_argument("start-maximized")
# Add a more robust User-Agent to mimic a real browser
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/108.0.0.0 Safari/537.36")


# Initialize WebDriver
try:
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
except Exception as e:
    print(f"Error initializing WebDriver: {e}")
    print("Please ensure you have an active internet connection or try reinstalling webdriver_manager.")
    exit() # Exit if WebDriver cannot be initialized

### Scraping Procedure

In [2]:
# Navigate to Youtube results
search_query = "tech reviews 2025"
formatted_search_query = search_query.replace(" ", "+")
# Correct Youtube URL. No need for filter for 2025 reviews specifically if you want general results.
search_url = f"https://www.youtube.com/results?search_query={formatted_search_query}"
print(f"Navigating to search URL: {search_url}")
driver.get(search_url)
time.sleep(3) # Give some time for initial page load

# Handle potential consent dialogs or pop-ups (more robust XPATH)
try:
    consent_button = WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable((By.XPATH, "//button[contains(@aria-label, 'Agree to the use of cookies and other data for the purposes described')] | //button[contains(@aria-label, 'Reject the use of cookies')] | //button[contains(@aria-label, 'Accept all')] | //*[normalize-space(text())='Accept all'] | //*[normalize-space(text())='I agree']"))
    )
    consent_button.click()
    print("Consent dialog handled.")
    time.sleep(2)
except Exception as e:
    print(f"No consent dialog or could not click: {e}, proceeding...")

# Scroll to load more videos (limited scroll for faster execution and getting relevant results)
# We will scroll as needed within the loop to find enough regular videos.
def scroll_to_bottom(driver, scroll_attempts=3):
    last_height = driver.execute_script("return document.documentElement.scrollHeight")
    for i in range(scroll_attempts):
        driver.execute_script("window.scrollTo(0, document.documentElement.scrollHeight);")
        time.sleep(2) # Give time for content to load
        new_height = driver.execute_script("return document.documentElement.scrollHeight")
        if new_height == last_height:
            print(f"Scrolled {i+1} times, no new content loaded.")
            break
        last_height = new_height
    print(f"Finished initial scrolling.")

scroll_to_bottom(driver, scroll_attempts=3)


data = []
video_idx = 0         # Index to track potential video elements on the page
processed_regular_videos = 0 # Counter for regular videos successfully processed
regular_video_cnt = 3

while processed_regular_videos < regular_video_cnt: # Continue until we have 5 regular videos
    try:
        # Re-locate video elements in case more loaded from scrolling
        # Ensure we are looking for the main video renderer elements
        videos = WebDriverWait(driver, 10).until(
            EC.presence_of_all_elements_located((By.TAG_NAME, "ytd-video-renderer"))
        )

        # If we have run out of new videos from current scroll, try scrolling more
        if video_idx >= len(videos):
            print("Reached end of current loaded videos. Attempting to scroll more...")
            current_videos_count = len(videos)
            scroll_to_bottom(driver, scroll_attempts=2) # Scroll a bit more
            # After scrolling, re-find all videos to update the list
            videos = WebDriverWait(driver, 10).until(
                EC.presence_of_all_elements_located((By.TAG_NAME, "ytd-video-renderer"))
            )
            if len(videos) == current_videos_count: # If no new videos loaded after scrolling
                print("No more new videos loaded after additional scrolling. Stopping.")
                break # Exit the loop if no new videos appear

        # Get the next potential video element
        video_element = videos[video_idx]
        video_idx += 1

        # Attempt to get the video link first to check if it's a Short
        try:
            title_link = video_element.find_element(By.XPATH, ".//*[@id='video-title']")
            video_url = title_link.get_attribute("href")
        except Exception as e:
            print(f"Could not find video title link for element at index {video_idx-1}: {e}. Skipping.")
            continue # Skip this element if no link is found

        # --- IMPORTANT CHANGE: Check for "shorts" in URL ---
        if video_url and "/shorts/" in video_url:
            print(f"Skipping video (index {video_idx-1}) - Identified as a YouTube Short by URL.")
            continue # Skip to the next video element

        # If it's not a Short, proceed to scrape
        processed_regular_videos += 1
        current_video_number = processed_regular_videos
        print(f"\n--- Processing Regular Video #{current_video_number} ---")

        # Get the video URL and append &autoplay=0
        if video_url:
            # Append &autoplay=0 to prevent the video from playing
            if "?" in video_url:
                video_url += "&autoplay=0"
            else:
                video_url += "?autoplay=0"
            print(f"Navigating to video URL: {video_url}")
            driver.get(video_url)
            time.sleep(3) # Wait for the video page to load

            # Fallback: Pause the video using JavaScript if it still plays
            try:
                driver.execute_script("document.querySelector('video').pause();")
                print("Video paused via JavaScript.")
            except:
                print("Could not pause video (video element may not exist yet or no video playing).")
        else:
            print(f"Video {current_video_number} - Could not retrieve video URL, skipping.")
            processed_regular_videos -= 1 # Decrement count as this wasn't a valid video
            # Go back to search results immediately if URL was missing
            driver.get(search_url)
            time.sleep(3)
            continue # Continue to the next iteration to find another video

        # Extract data from the video page
        video_data = {}

        # Extract title
        try:
            title_element = WebDriverWait(driver, 10).until(
                # EC.visibility_of_element_located((By.XPATH, "//*[@id='title']/h1/yt-formatted-string"))
                EC.visibility_of_element_located((By.XPATH, './/*[@id="title"]/h1/yt-formatted-string'))
            )
            video_data["video_title"] = title_element.text.strip()
            print(f"Title: {video_data['video_title']}")
        except Exception as e:
            print(f"Error extracting title: {e}")
            video_data["video_title"] = "N/A"

        # Extract channel name and link
        try:
            channel_element = WebDriverWait(driver, 10).until(
                EC.visibility_of_element_located((By.XPATH, "//*[@id='text']/a"))
            )
            video_data["channel_name"] = channel_element.text.strip()
            video_data["channel_url"] = channel_element.get_attribute("href")
            print(f"Channel: {video_data['channel_name']}, URL: {video_data['channel_url']}")
        except Exception as e:
            print(f"Error extracting channel name/URL: {e}")
            video_data["channel_name"] = "N/A"
            video_data["channel_url"] = "N/A"


        # Extract subscribers
        try:
            subscribers_element = WebDriverWait(driver, 10).until(
                EC.visibility_of_element_located((By.XPATH, "//*[@id='owner-sub-count']"))
            )
            video_data["subscribers"] = subscribers_element.text.strip()
            print(f"Subscribers: {video_data['subscribers']}")
        except Exception as e:
            print(f"Error extracting subscribers: {e}")
            video_data["subscribers"] = "N/A"

        # --- MODIFICATION START: Separate Views and Publish Date XPaths ---
        # Extract views
        try:
            views_element = WebDriverWait(driver, 10).until(
                EC.visibility_of_element_located((By.XPATH, "//*[@id='info']/span[1]"))
            )
            video_data["views"] = views_element.text.strip()
            print(f"Views: {video_data['views']}")
        except Exception as e:
            print(f"Error extracting views: {e}")
            video_data["views"] = "N/A"

        # Extract publish date
        try:
            publish_date_element = WebDriverWait(driver, 10).until(
                EC.visibility_of_element_located((By.XPATH, './/*[@id="info"]/span[3]'))
            )
            video_data["publish_date"] = publish_date_element.text.strip()
            print(f"Publish Date: {video_data['publish_date']}")
        except Exception as e:
            print(f"Error extracting publish date: {e}")
            video_data["publish_date"] = "N/A"
        # --- MODIFICATION END ---


        # Extract likes (this XPath is often fragile, YouTube changes this frequently)
        # try:
        #     # Wait for the element to be visible and its text content to be stable.
        #     # We can try to wait until the text doesn't change for a short period,
        #     # or look for a specific pattern (e.g., only digits and K/M/B for thousands/millions/billions).

        #     # First, wait until the element is visible
        #     likes_element = WebDriverWait(driver, 10).until(
        #         EC.visibility_of_element_located((By.XPATH, './/*[@id="top-level-buttons-computed"]/segmented-like-dislike-button-view-model/yt-smartimation/div/div/like-button-view-model/toggle-button-view-model/button-view-model/button/div[2]'))
        #     )

        #     # Now, wait for the text to stabilize. This is a common technique for animating numbers.
        #     # We'll poll the text for a short duration and check if it stops changing.
        #     initial_likes_text = likes_element.text.strip()
        #     stable_likes_text = initial_likes_text
        #     start_time = time.time()
        #     while time.time() - start_time < 2:  # Wait up to 2 seconds for text to stabilize
        #         time.sleep(0.1)  # Check every 0.1 seconds
        #         current_likes_text = likes_element.text.strip()
        #         if current_likes_text == stable_likes_text:
        #             break  # Text has stabilized
        #         stable_likes_text = current_likes_text

        #     video_data["likes"] = stable_likes_text
        #     print(f"Likes: {video_data['likes']}")
        # except Exception as e:
        #     print(f"Error extracting likes: {e}. (XPath for likes is very brittle on YouTube)")
        #     video_data["likes"] = "N/A"

        # Add the original search URL and video URL
        # video_data["search_url"] = search_url
        video_data["video_url"] = video_url.replace("&autoplay=0", "") # Store original URL without autoplay

        # Append the data
        data.append(video_data)

        # Go back to the search results
        driver.get(search_url)
        time.sleep(3) # Wait for the search results to reload

        # Scroll again to ensure subsequent videos are loaded
        scroll_to_bottom(driver, scroll_attempts=1) # Just one more scroll

    except Exception as e:
        print(f"An unexpected error occurred during video processing (index {video_idx-1}): {e}")
        # If an error occurs that prevents processing, we decrement processed_regular_videos
        # so the loop continues to try and find the required number of videos.
        if processed_regular_videos > 0: # Only decrement if we actually incremented it for this loop iteration
            processed_regular_videos -= 1
        # It's crucial to go back to the search page if an error occurred on the video page
        driver.get(search_url)
        time.sleep(3)
        scroll_to_bottom(driver, scroll_attempts=1) # Scroll to find next video

Navigating to search URL: https://www.youtube.com/results?search_query=tech+reviews+2025
No consent dialog or could not click: Message: 
Stacktrace:
	GetHandleVerifier [0x0078FC03+61635]
	GetHandleVerifier [0x0078FC44+61700]
	(No symbol) [0x005B05D3]
	(No symbol) [0x005F899E]
	(No symbol) [0x005F8D3B]
	(No symbol) [0x00640E12]
	(No symbol) [0x0061D2E4]
	(No symbol) [0x0063E61B]
	(No symbol) [0x0061D096]
	(No symbol) [0x005EC840]
	(No symbol) [0x005ED6A4]
	GetHandleVerifier [0x00A14523+2701795]
	GetHandleVerifier [0x00A0FCA6+2683238]
	GetHandleVerifier [0x00A2A9EE+2793134]
	GetHandleVerifier [0x007A68C5+155013]
	GetHandleVerifier [0x007ACFAD+181357]
	GetHandleVerifier [0x00797458+92440]
	GetHandleVerifier [0x00797600+92864]
	GetHandleVerifier [0x00781FF0+5296]
	BaseThreadInitThunk [0x76885D49+25]
	RtlInitializeExceptionChain [0x7796D03B+107]
	RtlGetAppContainerNamedObjectPath [0x7796CFC1+561]
, proceeding...
Finished initial scrolling.

--- Processing Regular Video #1 ---
Navigating to 

### Storing the data in a CSV file

In [ ]:
# Save data to a DataFrame
if data: # Only create DataFrame if data was collected
    df = pd.DataFrame(data)
    print("\nFinal DataFrame:")
    # print(df)
    # df
    df.to_csv("csvs/yt_video_data.csv", index=False, encoding='utf-8')
    print("Data saved to youtube_video_data.csv")
else:
    print("\nNo video data was collected.")


Final DataFrame:
                                         video_title channel_name  \
0  Best of Computex 2025: Top Tech Innovations Yo...        PCMag   
1                      The CRAZIEST Tech at CES 2025     Tussalty   
2                           35,000 TV vs 3,50,000 TV    TechWiser   

                          channel_url        subscribers       views  \
0      https://www.youtube.com/@PCMag   232K subscribers  4.6K views   
1   https://www.youtube.com/@Tussalty   123K subscribers   27K views   
2  https://www.youtube.com/@techwiser  2.35M subscribers  5.5M views   

   publish_date                                          video_url  
0     1 day ago  https://www.youtube.com/watch?v=2cMtBVtYsqk&pp...  
1  4 months ago  https://www.youtube.com/watch?v=3YuG6FTsYxE&pp...  
2    1 year ago  https://www.youtube.com/watch?v=xheRfrUhfaE&pp...  
Data saved to youtube_video_data.csv


### DataFrame View

In [6]:
records = pd.read_csv('yt_video_data.csv')
records

,video_title,channel_name,channel_url,subscribers,views,publish_date,video_url
0,Best of Computex 2025: Top Tech Innovations Yo...,PCMag,https://www.youtube.com/@PCMag,232K subscribers,4.6K views,1 day ago,https://www.youtube.com/watch?v=2cMtBVtYsqk&pp...
1,The CRAZIEST Tech at CES 2025,Tussalty,https://www.youtube.com/@Tussalty,123K subscribers,27K views,4 months ago,https://www.youtube.com/watch?v=3YuG6FTsYxE&pp...
2,"35,000 TV vs 3,50,000 TV",TechWiser,https://www.youtube.com/@techwiser,2.35M subscribers,5.5M views,1 year ago,https://www.youtube.com/watch?v=xheRfrUhfaE&pp...


### Closing the Web Driver

In [4]:
driver.quit()